# classification

In [7]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

random_state = 67
np.random.seed(random_state)
cv_splits = StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=random_state)

In [ ]:
target = ''
url = ''
df = pd.read_csv(url)


In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.shape[0]-df.dropna().shape[0]

In [ ]:
df.isna().sum()

In [ ]:
df[target].value_counts().sort_index().plot(kind='bar',rot=0)

In [ ]:
df.boxplot(figsize=(15,12))

In [ ]:
sns.pairplot(df,hue=target)

## Preprocessing

In [ ]:
X = df.drop(target,axis=1)
y = df[target]

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
mms = MinMaxScaler()
X_scaled = pd.DataFrame(mms.fit_transform(X),columns=df.columns)

from sklearn.preprocessing import PowerTransformer,StandardScaler
from sklearn.pipeline import make_pipeline
pipeline = make_pipeline(PowerTransformer(),StandardScaler())
X_normalized = pd.DataFrame(pipeline.fit_transform(X_scaled),columns=df.columns)

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.67,random_state=random_state)

### Training

In [ ]:
clfs = []
results = pd.DataFrame([],columns=['scoring','model','best_param','accuracy','precision_macro','recall_macro','f1_macro'])

In [ ]:
scorings = ['accuracy','precision_macro','recall_macro','f1_macro']

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier,RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import Perceptron
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

model_lbls = [
    'dt',
    'nb',
    'lp',
    'knn',
    #'svm',
    'rf',
    'adb'
]

param_dt = [{'max_depth':[*range(1,20)],'class_weight':[None,'balanced']}]
param_nb = [{'var_smoothing':[10*exp for exp in range(-3,-12,-1)]}]
param_lp = [{'early_stopping':[True,False],'class_weight':[None,'balanced']}]
param_knn = [{'n_neighbors':[*range(2,7)]}]
param_abd = [{'n_estimators':[*range(10,51,10)],'learning_rate':[0.5,0.75,1,1.25,1.5]}]
param_rf = [{'n_estimators':[*range(10,30,4)],'max_depth':[*range(4,30,4)],'class_weight':[None,'balanced']}]

param_svm = [
    {'kernel':['rbf'],'gamma':[1e-3,1e-4],'C':[1,10,100]},
    {'kernel':['linear'],'C':[1,10,100]},
]

models = {
    'dt':{
        'name': 'Decision Tree',
        'estimator': DecisionTreeClassifier(random_state=random_state),
        'param': param_dt
    },
    'nb':{
        'name': 'Naive Bayers',
        'estimator': GaussianNB(),
        'param': param_nb
    },
    'lp':{
        'name': 'Linear perceptron',
        'estimator': Perceptron(random_state=random_state),
        'param': param_lp
    },
    'knn':{
        'name': 'K nearest neighbors',
        'estimator': KNeighborsClassifier(),
        'param': param_knn
    },
    'svm':{
        'name': 'SVM',
        'estimator': SVC(random_state=random_state),
        'param': param_svm
    },
    'rf':{
        'name': 'Random Forest',
        'estimator': RandomForestClassifier(random_state=random_state),
        'param': param_rf
    },
    'adb':{
        'name': 'adaboost',
        'estimator': AdaBoostClassifier(random_state=random_state),
        'param': param_abd
    }
}

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

for score in scorings:
    for m in model_lbls:
        clf = GridSearchCV(
            estimator=models[m]['estimator'],
            param_grid=models[m]['param'],
            scoring=score,
            cv=cv_splits
        )
        clf.fit(X_train,y_train)
        clfs.append(clf.best_estimator_)
        y_pred = clf.predict(X_test)
        cr = classification_report(y_test,y_pred,output_dict=True,zero_division=1)
        results.loc[len(results)]=[
            score,
            models[m]['name'],
            clf.best_params_,
            cr['accuracy'],
            cr['macro avg']['precision'],
            cr['macro avg']['recall'],
            cr['macro avg']['f1-score']
        ]

## Display results

In [ ]:
for score in scorings:
    display(
        results[results.scoring==score]\
            .sort_values(by=score,ascending=False)\
            .drop('scoring',axis=1)\
            .style.format(precision=3)\
            .set_caption(f'Best Models for:{score}')
    )

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

for score in scorings:
    bestRow = results.loc[results.scoring==score,score].argmax(axis=0)
    best_clf = clfs[bestRow]
    disp = ConfusionMatrixDisplay.from_estimator(best_clf,X_test,y_test)
    disp.ax_.set_title(f'Confusion Matrix of {results.loc[bestRow]['model']} for score:{score}')